<div style="
    background: linear-gradient(90deg, #0f2027 0%, #2c5364 100%);
    padding: 32px 0 32px 0;
    border-radius: 15px;
    text-align: center;
    margin-bottom: 30px;
">
  <h1 style="color: #fff; letter-spacing: 2px; font-size: 2.7rem; margin: 0; font-weight: 800;">
    Module 2: Building Multimodal Data Pipeline (Part 1)
  </h1>
</div>


This notebook will provide an overview of Ray Data and how to use it to read, transform and write data in a distributed manner.

<div class="alert alert-block alert-info">
<b> Here is the roadmap for this notebook:</b>
<ul>
  <li>When and why to use Ray Data?</li>
  <li>How to work with Ray Data</li>
  <li>Loading data</li>
  <li>Lazy execution mode</li>
  <li>Transforming data</li>
  <li>Expressions API</li>
  <li>Materializing data</li>
  <li>Persisting data</li>
</ul>
</div>

**Imports**

In [ ]:
import pandas as pd
import ray

In [ ]:
READ_DATA_PATH = "s3://anyscale-public-materials/nyc-taxi-cab-features/year=2014/month=4/"

## 1. When to Consider Ray Data

Consider using Ray Data for your project if it meets one or more of the following criteria:


| **Challenge** | **Ray Data Solution** |
|---------------|------------------------|
| **Operating on large datasets** | Distributes data loading and processing across a Ray cluster to handle massive datasets efficiently |
| **Feeding data into distributed training** | Streams data to training processes with configurable batch sizes and supports parallelism across CPU and GPU resources |
| **Building reliable pipelines** | Leverages Ray Core's fault-tolerance mechanisms to recover from failures such as network errors, spot instance preemptions, and hardware faults |

## 2. How to work with Ray Data

It is commonly a three step process when using Ray Data:
1. Create your Dataset (most commonly using an IO connector)
2. Apply transformations to your Dataset
3. Consume your Dataset by either:
   1. Writing it out to a sink (file-based or database)
   2. Iterating over it (connecting it to a training process)

## 3. Loading data

Assume we have a parquet dataset on S3

In [ ]:
!aws s3 ls $READ_DATA_PATH --human-readable --recursive

We will use the `read_parquet` function to create a Dataset from the parquet files.

In [ ]:
ds = ray.data.read_parquet(READ_DATA_PATH, include_paths=True)
type(ds)

<div class="alert alert-block alert-info">
  <p><strong>Ray Data supports a variety of data sources for loading data</strong></p>
  <ul>
    <li>
      Reading files from common file formats (e.g. Parquet, CSV, JSON, etc.)
      <ul>
          <li><code>ds = ray.data.read_parquet("s3://...")</code></li>
      </ul>
    </li>
    <li>Loading from in-memory data structures (e.g. NumPy, PyTorch, etc.)
      <ul>
          <li><code>ray.data.from_torch(torch_ds)</code></li>
      </ul>
    <li>Loading from data lakehouses and warehouses such as Snowflake, Iceberg, and Databricks.</li>
      <ul>
          <li><code>ds = ray.data.read_databricks_tables()</code></li>
      </ul>
  </ul>
  <p>
    Start with an extensive list of <a href="https://docs.ray.io/en/latest/data/api/input_output.html#input-output" target="_blank">supported formats</a> and review further options in our <a href="https://docs.ray.io/en/latest/data/loading-data.html#loading-data" target="_blank">data loading guide</a>.
  </p>
</div>

Under the hood, Ray Data uses Ray tasks to read data from remote storage

|<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/Ray+Data+Internals+-+reading.png" width="500px" loading="lazy">|
|:--|
|When reading from a file-based datasource, Ray Data starts with a number of read tasks proportional to the number of CPUs in the cluster. |
|Each read task reads its assigned files and produces output blocks.|

### Note on blocks

|<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-intro/block.png" width="700px" loading="lazy">|
|:--|
|A Dataset when materialized is a distributed collection of blocks. This example illustrates a materialized dataset with three blocks, each block holding 1000 rows.|

<div class="alert alert-block alert-info">
<strong>Block</strong> is a contiguous subset of rows from a dataset. Blocks are distributed across the cluster and processed independently in parallel. By default blocks are PyArrow tables.
</div>

## 4. Lazy execution mode

In Ray Data, operations are not executed immediately. Most transformations are **lazy**, meaning they build up an execution plan rather than running right away. 

The execution plan is only **executed** when you call a method that *materializes* or *consumes* the dataset.

To materialize a small subset of the data, you can use the `take_batch` method.

In [ ]:
batch = ds.take_batch(batch_size=3)
batch

<div class="alert alert-block alert-info">

<b>Note on execution triggering methods in Ray Dataset</b>

To determinte if an operation will trigger execution, look for the methods with the `ConsumptionAPI` decorator in the [`Dataset.py`](https://github.com/ray-project/ray/blob/master/python/ray/data/dataset.py).

These categories of operations trigger execution (with some examples):
* Method designed to consume Datasets for **writing**:
  * [`write_parquet`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.write_parquet.html#ray.data.Dataset.write_parquet)
* Method designed to consume Datasets for **distributed training**:
  * [`streaming_split`](https://github.com/ray-project/ray/blob/master/python/ray/data/dataset.py#L1694)
* Methods that attempt to **show** data, for example:
  * [`take`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.take.html#ray.data.Dataset.take)
  * [`show`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.show.html#ray-data-dataset-show)
* **Aggregations**, which attempt to reduce a dataset to a single value per column:
  * [`min`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.min.html#ray.data.Dataset.min)
  * [`sum`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.sum.html#ray.data.Dataset.sum)

Another way to trigger execution is to explicitly call <a href="https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.materialize.html#ray-data-dataset-materialize" target="_blank">materialize()</a>. This will execute the underlying plan and generate the entire data blocks onto the cluster's memory.

## 5. Transforming data

To transform data, we can use the [`map_batches`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.map_batches.html#ray-data-dataset-map-batches) API. 

`map_batches` takes a user-defined function which accepts a batch of data and returns a batch of transformed data.

In [ ]:
def normalize(batch_df: pd.DataFrame) -> pd.DataFrame:
    batch_df["fare_pct"] = batch_df["fare_amount"] / batch_df["total_amount"] * 100
    return batch_df

Calling `ds.map_batches` will add a `map_batches` operator to the execution plan.

In [ ]:
ds_normalized = ds.map_batches(normalize)

To tune the batch size for the transformation, specify the `batch_size` parameter.  

In [ ]:
ds_normalized = ds.map_batches(normalize, batch_size=32)

<div class="alert alert-block alert-info">

**Note:** batching only helps with performance when a transformation is vectorized - i.e. benefits from processing multiple rows at once.

Finding the optimal batch size depends on the hardware available and the target utilization.

</div>

### 5.1 On resource specification

To specify the exact resources to use for a `map_batches` transformation, specify the `num_cpus`, `num_gpus`, `memory`, and `resources` parameters.

- `num_cpus`: Number of CPUs to use for each task (use >1 if task performs multithreaded operations).
- `num_gpus`: Number of GPUs to use for each task.
- `memory`: Amount of RAM to use for each task (in bytes).
- `resources`: What is referred to as custom resources in Ray. It is a way to specify which node types to use for each task.

In [ ]:
ds_normalized = ds.map_batches(normalize, num_cpus=1, memory=1 * 1024**3)

<div class="alert alert-block alert-info">

**Note:** Ray only performs a logical allocation of resources and does not physically enforce resource limits.

By default, Ray will [retry OOM errors](https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html#retry-policy) and Ray Data will infinitely retry tasks that fail due to system failures.

Specifying resources helps avoid resource contention, avoiding unnecessary retries and confusing OOM errors.

</div>

To verify the output of `normalize()`, call [`take_batch()`](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.take_batch.html#ray.data.Dataset.take_batch) on the dataset.

In [ ]:
normalized_batch = ds_normalized.take_batch(batch_size=10)

Let's inspect the normalized_batch's columns

In [ ]:
pd.DataFrame(
    {
        "fare_amount": normalized_batch["fare_amount"],
        "total_amount": normalized_batch["total_amount"],
        "fare_pct": normalized_batch["fare_pct"],
    }
)

### 5.2 Activity: Implement a custom transformation

<div class="alert alert-block alert-info">

Here is what you need to do:

1. Add a column named "is_cash_payment" that is True if the payment_type equals 2, otherwise False.
2. Inspect the output of the transformation.


<details>
<summary>Click to view hints</summary>

Starting point:

```python
# Hint: Implement the add_label function
def add_payment_type_label(batch: pd.DataFrame) -> pd.DataFrame:
    ...
    return batch

# Hint: Use `map_batches` to apply the add_label function
ds_labeled = ds_normalized.map_batches(add_payment_type_label)

# Hint: Take a batch of the labeled dataset
labeled_batch = ds_labeled.take_batch(10)
print(labeled_batch)
```

</details>

</div>

In [ ]:
# Write your solution here

<div class="alert alert-block alert-info">

<details>

<summary>Click to view solution</summary>

```python
def add_payment_type_label(batch: pd.DataFrame) -> pd.DataFrame:
    batch["is_cash_payment"] = batch["payment_type"] == 2
    return batch

ds_labeled = ds_normalized.map_batches(add_payment_type_label)
labeled_batch = ds_labeled.take_batch(10)
print(labeled_batch)
```

</details>  
</div>

## 6. (New) Ray Data Expressions

Ray Data Expressions are a new way to transform data in Ray Data.

When using expressions, Ray Data should be more efficient given it can optimize the execution plan better. 

In [ ]:
from ray.data.expressions import col

ds_transformed = ds_normalized.with_column("face_portion", col("fare_pct") / 100)

For more complex transformations, you can use the `udf` decorator to define a function that can be used in the expression.

In [ ]:
from ray.data.expressions import udf, DataType
import pyarrow.compute as pc

# instead of an open-ended map_batches this clearly identifies the required inputs and outputs
@udf(return_dtype=DataType.float32())
def divide(column1, column2):
    # Best to return a pyarrow array
    return pc.divide(column1, column2)

ds_transformed_2 = ds_transformed.with_column("face_portion", divide(col("fare_amount"), col("total_amount")))

<div class="alert alert-block alert-info">

**NOTE**: Use expressions for simple transformations, or transformations that you are willing to rewrite with pyarrow.

</div>

To see, how expressions help, we can inspect how Ray Data plans and optimizes execution using the `explain()` method.

In [ ]:
(
    ray.data.range(1e6)
    .with_column("id_doubled", col("id") * 2)
    .filter(expr=col("id") < 3)
    .explain()
)

In the above case you can see how Ray Data clearly pushed down the filter given it knows the id column will still be in the dataset after the column expresssion

In [ ]:
def double_id(batch):
    batch["id"] = batch["id"] * 2
    return batch

(
    ray.data.range(1e6)
    .map_batches(double_id, batch_format="pandas")
    .filter(expr=col("id") < 3)
    .explain()
)

In the `map_batches` case, Ray Data will have to compute all 1 million rows first before applying the filter

## 7. Materializing data

You can choose to materialize the entire dataset into the Ray object store which is distributed across the cluster, primarily in memory and secondarily spilling to disk.

To materialize the dataset, we can use the `materialize()` method.

Use this **only** when you require the full dataset to compute downstream outputs.

In [ ]:
# Set the name of a pipeline prior to execution to find easily in dashboards
ds_transformed_2.set_name("transform_dataset")
ds_transformed_2 = ds_transformed_2.materialize()

`materialize()` triggers the execution. The logs should show the execution plan of Dataset:

```
Execution plan of Dataset: InputDataBuffer[Input] -> TaskPoolMapOperator[ListFiles] -> TaskPoolMapOperator[ReadFiles] -> TaskPoolMapOperator[MapBatches(normalize)] -> ActorPoolMapOperator[MapBatches(MNISTClassifier)]
```

## 8. Persisting data

Finally, you can persist a dataset to storage using any of the "write" functions that Ray Data supports.

Lets write our predictions to a parquet dataset.

In [ ]:
ds_transformed_2.set_name("write_transformed_dataset")
ds_transformed_2.write_parquet("/mnt/cluster_storage/transformed_data")

Refer to the [Input/Output docs](https://docs.ray.io/en/latest/data/api/input_output.html) for a comprehensive list of write functions.